# Bronze — Auto Loader (Orders)

Ingest `olist_orders_dataset.csv` using **Auto Loader** into `globalmart.bronze.orders_autoloader`.

**Idempotency:** Auto Loader tracks processed files in a **checkpoint** — re-running on the same files adds **zero rows**.

Compare with batch ingestion (`01_idempotent_ingestion.ipynb`) which uses a **file fingerprint metadata table**.

In [ ]:
import sys

notebook_path = (
    dbutils.notebook.entry_point.getDbutils()
    .notebook()
    .getContext()
    .notebookPath()
    .get()
)
REPO_ROOT = notebook_path.split("/notebooks/")[0]
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

from src.ingestion.auto_loader import AutoLoaderConfig, stage_orders_file, run_orders_autoloader

config = AutoLoaderConfig()
print("Source:", config.source_path)
print("Target:", config.target_table)
print("Checkpoint:", config.checkpoint_path)

In [ ]:
# Stage orders CSV into Auto Loader input folder
stage_orders_file(dbutils, config)
display(spark.createDataFrame(
    [(f.name, f.size) for f in dbutils.fs.ls(config.source_path)],
    ["file", "bytes"],
))

In [ ]:
# First Auto Loader run
run_1 = run_orders_autoloader(spark, config)
print(run_1)
display(spark.table(config.target_table).limit(5))

In [ ]:
# Second run — idempotency proof (expect rows_added = 0)
run_2 = run_orders_autoloader(spark, config)
print(run_2)
print(f"Rows added on second run: {run_2['rows_added']}")

In [ ]:
# Compare with batch bronze.orders
batch_count = spark.table("globalmart.bronze.orders").count()
auto_count = spark.table(config.target_table).count()
print(f"Batch orders table:     {batch_count:,} rows")
print(f"Auto Loader table:      {auto_count:,} rows")
print(f"Match: {batch_count == auto_count}")